In [6]:
import kafou_arraylake as arraylake
import zarr
import numpy as np
import xarray as xr

repo_name = "kafou/aurora-era5-samples"
branch = "extend-2025"

client = arraylake.Client()
repo = client.get_repo(repo_name)
ro = repo.readonly_session(branch)
ds = xr.open_zarr(
    ro.store,
    group="samples",
    zarr_format=3,
    consolidated=False,
    chunks=None,
)

ds



<xarray.Dataset> Size: 36TB
Dimensions:       (atmos_levels: 13, time: 124548, channel: 69, latitude: 721,
                   longitude: 1440)
Coordinates:
  * atmos_levels  (atmos_levels) int64 104B 50 100 150 200 ... 700 850 925 1000
  * longitude     (longitude) float64 12kB 0.0 0.25 0.5 ... 359.2 359.5 359.8
  * time          (time) datetime64[ns] 996kB 1940-01-01 ... 2025-03-31T18:00:00
  * latitude      (latitude) float64 6kB 90.0 89.75 89.5 ... -89.5 -89.75 -90.0
Dimensions without coordinates: channel
Data variables:
    sample_data   (time, channel, latitude, longitude) float32 36TB ...
Attributes:
    var_locs:  {'sfc': {'2t': [0, 1], 'msl': [1, 1], '10u': [2, 1], '10v': [3...

In [1]:
import os

os.environ["HF_HUB_DISABLE_SSL_VERIFICATION"] = "1"
os.environ["CURL_CA_BUNDLE"] = ""
os.environ["REQUESTS_CA_BUNDLE"] = ""

In [4]:
import numpy as np
import xarray as xr
import datetime

init_time = datetime.datetime(2025, 1, 1, 0)
steps = 5
S, F = 259200, 4  # example dimensions

times = [init_time + datetime.timedelta(hours=6*(i+1)) for i in range(steps)]
lv_arr = np.random.randn(steps, S, F).astype("float32")

lv_ds = xr.Dataset(
    coords={"time": ("time", times)},
    data_vars={"lv": (("time", "spatial_location", "feature"), lv_arr)},
)

print(lv_ds)


<xarray.Dataset> Size: 21MB
Dimensions:  (time: 5, spatial_location: 259200, feature: 4)
Coordinates:
  * time     (time) datetime64[ns] 40B 2025-01-01T06:00:00 ... 2025-01-02T06:...
Dimensions without coordinates: spatial_location, feature
Data variables:
    lv       (time, spatial_location, feature) float32 21MB 0.4066 ... 0.08372


In [8]:
from forecast_latent_vector_writer  import detect_latent_dim_from_rollout, reshape_rollout_to_cube, write_metadata


detect_latent_dim_from_rollout(lv_ds)

reshape_rollout_to_cube(lv_ds, init_time)




[DETECT] spatial_location=259200, feature=4 => latent_dim=1036800

[RESHAPE] reshape_rollout_to_cube
[RESHAPE] init_time=2025-01-01 00:00:00
[RESHAPE] lv raw shape=(5, 259200, 4) => lv_flat shape=(5, 1036800)
[RESHAPE] cube sizes=Frozen({'init_time': 1, 'lead_time': 5, 'lv': 1036800})


<xarray.Dataset> Size: 21MB
Dimensions:          (init_time: 1, lead_time: 5, lv: 1036800)
Coordinates:
  * init_time        (init_time) datetime64[us] 8B 2025-01-01
  * lead_time        (lead_time) int64 40B 6 12 18 24 30
    valid_time       (init_time, lead_time) datetime64[us] 40B 2025-01-01T06:...
Dimensions without coordinates: lv
Data variables:
    latent_forecast  (init_time, lead_time, lv) float32 21MB 0.4066 ... 0.08372

In [3]:
import datetime
import numpy as np
import xarray as xr

steps = 5
n_spatial = 4 * 180 * 360   # 259200
n_feat = 24

init_times = [
    datetime.datetime(2025, 1, 1, 0),
    datetime.datetime(2025, 1, 1, 6),
]

rollouts = []

# --------------------------------------------------
# Fake rollouts (time, spatial_location, feature)
# --------------------------------------------------
for init_time in init_times:
    lv = np.random.randn(steps, n_spatial, n_feat).astype("float32")

    times = [
        init_time + datetime.timedelta(hours=6 * (i + 1))
        for i in range(steps)
    ]

    lv_ds = xr.Dataset(
        coords={"time": ("time", times)},
        data_vars={
            "lv": (("time", "spatial_location", "feature"), lv),
        },
    )

    rollouts.append((init_time, lv_ds))


# --------------------------------------------------
# Reshape rollout → forecast cube (FLAT)
# --------------------------------------------------
def reshape_rollout_flat(
    lv_ds: xr.Dataset,
    init_time: datetime.datetime,
):
    lv = lv_ds["lv"].values  # (steps, spatial_location, feature)
    steps, S, F = lv.shape

    init_np = np.datetime64(init_time)
    times = lv_ds["time"].values
    lead_hours = ((times - init_np) / np.timedelta64(1, "h")).astype("int64")

    ds = xr.Dataset(
        data_vars={
            "latent_forecast": (
                ("init_time", "lead_time", "spatial_location", "feature"),
                lv[None, ...],
            )
        },
        coords={
            "init_time": ("init_time", [init_np]),
            "lead_time": ("lead_time", lead_hours),
            "spatial_location": ("spatial_location", np.arange(S)),
            "feature": ("feature", np.arange(F)),
        },
    )

    valid_time = init_np + lead_hours * np.timedelta64(1, "h")
    ds = ds.assign_coords(
        valid_time=(("init_time", "lead_time"), valid_time[None, :])
    )

    return ds


# --------------------------------------------------
# Build full dataset (2 init_times)
# --------------------------------------------------
cubes = []

for init_time, lv_ds in rollouts:
    cubes.append(reshape_rollout_flat(lv_ds, init_time))

cube_all = xr.concat(cubes, dim="init_time")

print(cube_all)


<xarray.Dataset> Size: 251MB
Dimensions:           (init_time: 2, lead_time: 5, spatial_location: 259200,
                       feature: 24)
Coordinates:
  * init_time         (init_time) datetime64[us] 16B 2025-01-01 2025-01-01T06...
  * lead_time         (lead_time) int64 40B 6 12 18 24 30
  * spatial_location  (spatial_location) int64 2MB 0 1 2 ... 259198 259199
  * feature           (feature) int64 192B 0 1 2 3 4 5 6 ... 18 19 20 21 22 23
    valid_time        (init_time, lead_time) datetime64[us] 80B 2025-01-01T06...
Data variables:
    latent_forecast   (init_time, lead_time, spatial_location, feature) float32 249MB ...
